# LLM Ágens érdekes ágensek személyiségei (demó)

A demo célja egy olyan automatizált technológiai híradó rendszer demonstrálása, amely a egy választott nyelvi modell tesztelése különböző személyiségekkel és ágens képességével együtt. A folyamat során beállításra kerültek a szükséges SDK-kat, definiáltam egy JSON-sémát a strukturált adatkinyeréshez, és integráltam a francia fejlesztésű Mistral modell `web_search` eszközét, hogy a modell valós idejű információk alapján dolgozhasson.

Az eredményt egy funkcionális Gradio webes felületen jelenik meg, amely személyiség kiválasztása után gombnyomásra összegyűjti, rendszerezi és formázott HTML-listaként megjeleníti a legfrissebb informatikai és technológiai híreket. A rendszer képessé vált a nyers AI-válaszok strukturált feldolgozására, biztosítva a forráshivatkozások és a témakörök szerinti kategorizálás pontosságát.

In [ ]:
!pip install mistralai gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.38.0
    Uninstalling opentelemetry-api-1.38.0:
      Successfully uninstalled opentelemetry-api-1.38.0
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.59b0
    Uninstalling opentelemetry-semantic-conventions-0.59b0:
      Successfully uninstalled opentelemetry-semantic-conventions-0.59b0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.28.0 requires opentelemetr

In [ ]:
import os
import httpx
from datetime import datetime
from mistralai.client import Mistral

# Aktuális dátum lekérése
today = datetime.now().strftime('%Y-%m-%d')

# API kulcs beállítása
api_key = os.environ.get("MISTRAL_API_KEY")

# Egyedi HTTP kliens létrehozása megnövelt időtúllépéssel (timeout)
# Ez segít, ha a webes keresés és a JSON generálás több időt vesz igénybe
custom_client = httpx.Client(timeout=120.0)
client = Mistral(api_key=api_key, client=custom_client)

inputs = [
    {
        "role": "user",
        "content": f"Keress rá a legfrissebb technológiai hírekre a mai napon ({today}), és foglald össze őket."
    }
]

completion_args = {
    "temperature": 0.45,
    "max_tokens": 4096,
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "response_schema",
            "schema": {
                "type": "object",
                "title": "StructuredData",
                "required": ["news_items"],
                "properties": {
                    "news_items": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "required": ["news_title", "news_date", "news_content", "news_topics", "news_url"],
                            "properties": {
                                "news_url": {"type": "string", "format": "uri"},
                                "news_date": {"type": "string"},
                                "news_title": {"type": "string"},
                                "news_topics": {"type": "string"},
                                "news_content": {"type": "string"}
                            }
                        }
                    }
                }
            }
        }
    },
}

tools = [{"type": "web_search"}]

try:
    response = client.beta.conversations.start(
        inputs=inputs,
        model="mistral-medium-latest",
        instructions=f"Híradós szerkesztő vagy. A mai dátum: {today}. Feladatod az aktuális hírek összefoglalása magyarul egy 'news_items' listába.",
        completion_args=completion_args,
        tools=tools
    )
    print("Sikeres lekérés!")
    print(response)
except Exception as e:
    print(f"Hiba történt a lekérés során: {e}")

Sikeres lekérés!
conversation_id='conv_019d5eb56f0a73cda4ed029b036295eb' outputs=[ToolExecutionEntry(name='web_search', arguments='{"query": "legfrissebb technológiai hírek", "start_date": "2026-04-05", "end_date": "2026-04-05"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 4, 5, 17, 34, 2, 227170, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 4, 5, 17, 34, 3, 821584, tzinfo=TzInfo(0)), agent_id=Unset(), model='mistral-medium-latest', id='tool_exec_019d5eb56ff37369beb86ffb543d44dd', info={}), MessageOutputEntry(content='{"news_items": [{"news_url": "https://index.hu/24ora/?cimke=technol%C3%B3gia", "news_date": "2026-04-05", "news_title": "Technológiai átverések és áttörések", "news_topics": "Technológia, AI, kripto", "news_content": "A nagy nyugati cégek egy még nem működő technológiát árusítanak. Palkovics László szerint ami technológiai kihívást jelent, az most nemzeti érdek is egyben. A Knorr-Bremse Budapest fejlesztőközpontjában a technológia

In [ ]:
import json

for entry in response.outputs:
    if entry.type == 'message.output':
        try:
            data = json.loads(entry.content)
            print(f"--- {len(data['news_items'])} hír érkezett ---\n")

            for i, item in enumerate(data['news_items'], 1):
                print(f"{i}. {item['news_title'].upper()}")
                print(f"Dátum: {item['news_date']}")
                print(f"Témák: {item['news_topics']}")
                print(f"Forrás: {item['news_url']}")
                print(f"{item['news_content']}\n" + "-"*30 + "\n")

        except (json.JSONDecodeError, KeyError) as e:
            print(f"Hiba a feldolgozás során: {e}")

--- 5 hír érkezett ---

1. TECHNOLÓGIAI ÁTVERÉSEK ÉS ÁTTÖRÉSEK
Dátum: 2026-04-05
Témák: Technológia, AI, kripto
Forrás: https://index.hu/24ora/?cimke=technol%C3%B3gia
A nagy nyugati cégek egy még nem működő technológiát árusítanak. Palkovics László szerint ami technológiai kihívást jelent, az most nemzeti érdek is egyben. A Knorr-Bremse Budapest fejlesztőközpontjában a technológia jelene már a jövőt képviseli.
------------------------------

2. ÚJ TECHNOLÓGIAI TERMÉKEK ÉS FEJLESZTÉSEK
Dátum: 2026-04-05
Témák: Tech, Apple, 3D nyomtatás
Forrás: https://www.hirguru.hu/tech
Az Apple kiadta az első iOS 18 és iPadOS 18 nyilvános bétaverzióit. A Prusa bemutatta az új Core One 3D nyomtatót. A Nike Air Max 1000 szinte teljes egészében 3D nyomtatott. A COBOD piacra dobja a BOD3 3D épületnyomtatót.
------------------------------

3. ADATGYŰJTÉS ÉS AI FEJLESZTÉSEK
Dátum: 2026-04-05
Témák: Infotech, AI, LinkedIn
Forrás: https://www.hirstart.hu/infotech.php
A LinkedIn titokban gyűjti az adatokat. Kí

[Gradio UI framework](https://www.gradio.app/guides/quickstart)

In [ ]:
import gradio as gr
import json

def get_tech_news(persona_name):
    # Személyiségek definiálása
    personas = {
        "Technokrata Futurista": "Optimista, adatközpontú, a technológiai fejlődésre és hatékonyságra fókuszáló hírszerkesztő.",
        "Szkeptikus Nyomozó": "Száraz, kritikus, minden állítást megkérdőjelező hírszerkesztő, aki a PR mögötti adatokat keresi.",
        "Bulvár-Guru": "Szenzációhajhász, rövid mondatokkal operáló, érzelmekre ható hírszerkesztő.",
        "Filozófus Elemző": "Megfontolt, kontextusba helyező hírszerkesztő, aki az etikai és társadalmi következményeket vizsgálja.",
        "Szatírikus Kritikus": "Humoros, ironikus hírszerkesztő, aki rávilágít a rendszer és a hírek abszurditásaira."
    }

    selected_instruction = personas.get(persona_name, "Általános hírszerkesztő.")

    gr.Info(f"A hírek lekérése folyamatban ({persona_name} stílusban)! Ez egy kis időt vehet igénybe.")

    try:
        inputs = [
            {
                "role": "user",
                "content": "Gyűjtsd össze a mai legfontosabb híreket külön-külön kifejtve!"
            }
        ]

        # Az instrukciót kiegészítjük a választott személyiséggel
        full_instructions = f"{selected_instruction} A híreket egy 'news_items' nevű listába rendezd a megadott séma szerint."

        response = client.beta.conversations.start(
            inputs=inputs,
            model="mistral-medium-latest",
            instructions=full_instructions,
            completion_args=completion_args,
            tools=tools
        )

        output_html = ""
        for entry in response.outputs:
            if entry.type == 'message.output':
                data = json.loads(entry.content)
                for item in data['news_items']:
                    # Háttérszín módosítva világosszürkére (#f0f0f0)
                    output_html += f"<div style='border: 1px solid #ddd; padding: 15px; margin-bottom: 15px; border-radius: 8px; background-color: #f0f0f0;'>"
                    output_html += f"<h3 style='color: #2c3e50;'>{item['news_title']}</h3>"
                    output_html += f"<p style='color: #7f8c8d; font-size: 0.9em;'>📅 <b>Dátum:</b> {item.get('news_date', 'N/A')}</p>"
                    output_html += f"<p style='color: #7f8c8d;'><b>Témák:</b> {item['news_topics']}</p>"
                    output_html += f"<div style='margin: 10px 0;'>{item['news_content']}</div>"
                    output_html += f"<a href='{item['news_url']}' target='_blank' style='color: #3498db; text-decoration: none;'>Forrás megnyitása &rarr;</a>"
                    output_html += "</div>"
        return output_html
    except Exception as e:
        return f"Hiba történt: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📰 Személyre Szabott Híradó")
    gr.Markdown("Válaszd ki a szerkesztő stílusát, majd kattints a gombra a legfrissebb hírekért.")

    with gr.Row():
        persona_dropdown = gr.Dropdown(
            choices=["Technokrata Futurista", "Szkeptikus Nyomozó", "Bulvár-Guru", "Filozófus Elemző", "Szatírikus Kritikus"],
            value="Technokrata Futurista",
            label="Szerkesztői személyiség"
        )

    btn = gr.Button("Hírek lekérése", variant="primary")
    output = gr.HTML(label="Hírek")

    btn.click(fn=get_tech_news, inputs=persona_dropdown, outputs=output)

demo.launch(debug=True)

/tmp/ipykernel_1184/538587284.py:54: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3b4595ad514cbe95b8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 